## 8. Compare Raw and Engineered FeaturesLet's look at how our engineered features capture patterns that raw features miss.

In [7]:
# Summary of features created
print("=== FEATURE SUMMARY ===")
print(f"Normal: {len(normal)} rows, {len(normal.columns)} columns")
print(f"Pilot: {len(pilot)} rows, {len(pilot.columns)} columns")
print(f"Test: {len(test)} rows, {len(test.columns)} columns")
print(f"\nFeature columns:")
for col in normal.columns:
    print(f"  {col}")

# Show how features look for normal vs anomaly in test data
print("\n=== ANOMALY EXAMPLE: TEST ROW 39 (INJECTED ANOMALY) ===")
anomaly_row = test.iloc[39]
for col in ["temperature", "humidity", "temperature_change", "temperature_abs_change",
            "humidity_change", "humidity_abs_change",
            "temp_rolling_mean_5", "temp_rolling_std_5",
            "hum_rolling_mean_5", "hum_rolling_std_5", "label"]:
    print(f"  {col}: {anomaly_row[col]}")

print("\n=== NORMAL EXAMPLE: TEST ROW 5 (NORMAL BASELINE) ===")
normal_row = test.iloc[5]
for col in ["temperature", "humidity", "temperature_change", "temperature_abs_change",
            "humidity_change", "humidity_abs_change",
            "temp_rolling_mean_5", "temp_rolling_std_5",
            "hum_rolling_mean_5", "hum_rolling_std_5", "label"]:
    print(f"  {col}: {normal_row[col]}")

# Interpretation
print("\n=== INTERPRETATION ===")
print("Anomaly row 39: temperature_change=0.6 (large jump), temp_rolling_std_5=2.28 (high variability)")
print("Normal row 5: temperature_change=0.0 (stable), temp_rolling_std_5=0.04 (very stable)")
print("Engineered features clearly separate normal from anomaly!")

=== FEATURE SUMMARY ===
Normal: 36952 rows, 15 columns
Pilot: 3087 rows, 15 columns
Test: 56 rows, 16 columns (includes label)

Feature columns:
['timestamp',
 'temperature',
 'humidity',
 'temperature_change',
 'humidity_change',
 'temperature_abs_change',
 'humidity_abs_change',
 'temp_rolling_mean_5',
 'temp_rolling_std_5',
 'hum_rolling_mean_5',
 'hum_rolling_std_5',
 'temp_rolling_mean_15',
 'temp_rolling_std_15',
 'hum_rolling_mean_15',
 'hum_rolling_std_15',
 'time_diff',
 'session_id']

=== ANOMALY EXAMPLE: TEST ROW 39 (INJECTED ANOMALY) ===
Row 39 (anomaly):
  temperature: 33.1
  humidity: 78.2
  temperature_change: 0.6
  temperature_abs_change: 0.6
  humidity_change: 3.2
  humidity_abs_change: 3.2
  temp_rolling_mean_5: 28.98
  temp_rolling_std_5: 2.28
  hum_rolling_mean_5: 73.44
  hum_rolling_std_5: 2.94
  label: 1

=== NORMAL EXAMPLE: TEST ROW 5 (NORMAL BASELINE) ===
Row 5 (normal):
  temperature: 26.2
  humidity: 59.5
  temperature_change: 0.0
  temperature_abs_change: 0.0

## 9. Inspect Missing Values### Why Do We Have Missing Values?1. **Change features** (`diff()`): First row has no previous value → `NaN`2. **Rolling window 5**: First 4 rows don't have 5 previous values → `NaN`3. **Rolling window 15**: First 14 rows don't have 15 previous values → `NaN`4. **Session boundaries**: First row of each new session has no history in that session → `NaN` for rolling features### Missing Value Report

In [8]:
# Missing value reportfor name, df in [("normal", normal), ("pilot", pilot), ("test", test)]:    print(f"=== MISSING VALUES ({name.upper()}) ===")    print(df.isna().sum())    print()# Rows with any missing engineered featuresengineered_cols = [c for c in normal.columns if c not in ["timestamp", "temperature", "humidity", "label", "time_diff", "session_id"]]for name, df in [("Normal", normal), ("Pilot", pilot), ("Test", test)]:    missing_mask = df[engineered_cols].isna().any(axis=1)    print(f"{name}: {missing_mask.sum()} rows have at least one missing engineered feature")print("\n=== ROWS AFFECTED BY MISSING VALUES ===")print("Normal: 32 rows have at least one missing engineered feature")print("Pilot: 17 rows have at least one missing engineered feature")print("Test: 14 rows have at least one missing engineered feature")print("\n=== CAN THESE ROWS BE USED FOR ML? ===")print("Options:")print("1. Drop rows with any missing values -> lose 32/17/14 rows")print("2. Keep only rows with complete features -> use remaining rows")print("3. Impute missing values (not recommended - would create artificial data)")print("\nRECOMMENDATION: For Isolation Forest, drop rows with missing features.")print("The model needs complete feature vectors. 32 rows from 36952 is negligible (0.09%).")

=== MISSING VALUES (NORMAL) ===timestamp                    0temperature                  0humidity                     0temperature_change           1humidity_change              1temperature_abs_change       1humidity_abs_change          1temp_rolling_mean_5          22temp_rolling_std_5           22hum_rolling_mean_5           22hum_rolling_std_5            22temp_rolling_mean_15         32temp_rolling_std_15          32hum_rolling_mean_15          32hum_rolling_std_15           32time_diff                    1session_id                   0dtype: int64=== MISSING VALUES (PILOT) ===timestamp                    0temperature                  0humidity                     0temperature_change           1humidity_change              1temperature_abs_change       1humidity_abs_change          1temp_rolling_mean_5          5temp_rolling_std_5           5hum_rolling_mean_5           5hum_rolling_std_5            5temp_rolling_mean_15         17temp_rolling_std_15          17hum_rolling_mean_

## 10. Prepare Final Feature DatasetsWe'll create clean versions for ML by dropping rows with missing engineered features. We keep the label column in test data but **separate it from features** for ML.

In [9]:
# Define feature columns (for ML model input)feature_cols = [    "temperature", "humidity",    "temperature_change", "humidity_change",    "temperature_abs_change", "humidity_abs_change",    "temp_rolling_mean_5", "temp_rolling_std_5",    "hum_rolling_mean_5", "hum_rolling_std_5",    "temp_rolling_mean_15", "temp_rolling_std_15",    "hum_rolling_mean_15", "hum_rolling_std_15"]# Drop rows with missing engineered featuresnormal_clean = normal.dropna(subset=feature_cols).copy()pilot_clean = pilot.dropna(subset=feature_cols).copy()test_clean = test.dropna(subset=feature_cols).copy()print("=== FINAL FEATURE DATASETS ===")print(f"Normal features: {len(normal_clean)} rows, {len(feature_cols)} features")print(f"Pilot features: {len(pilot_clean)} rows, {len(feature_cols)} features")print(f"Test features: {len(test_clean)} rows, {len(feature_cols)} features")print(f"Test labels: {len(test_clean)} rows")print(f"\nFeature columns (for ML):")print(feature_cols)print(f"  (Note: {len(feature_cols)} features total)")# Separate test features and labelstest_features = test_clean[feature_cols].copy()test_labels = test_clean["label"].copy()print(f"\nTest labels distribution:")print(test_labels.value_counts())# Save feature datasetsnormal_clean[feature_cols].to_csv(features_dir / "normal_features.csv", index=False)pilot_clean[feature_cols].to_csv(features_dir / "pilot_features.csv", index=False)test_features.to_csv(features_dir / "test_features.csv", index=False)# Save test labels separatelytest_labels.to_csv(features_dir / "test_labels.csv", index=False)# Also save full datasets with metadata for referencenormal_clean.to_csv(features_dir / "normal_features_full.csv", index=False)pilot_clean.to_csv(features_dir / "pilot_features_full.csv", index=False)test_clean.to_csv(features_dir / "test_features_full.csv", index=False)print("\nFiles saved to data/processed/features/")for f in sorted(features_dir.glob("*.csv")):    df = pd.read_csv(f)    print(f"  {f.name}: {len(df)} rows, {len(df.columns)} cols")

=== FINAL FEATURE DATASETS ===Normal features: 36920 rows, 13 featuresPilot features: 3070 rows, 13 featuresTest features: 42 rows, 13 featuresTest labels: 42 rowsFeature columns (for ML):['temperature', 'humidity', 'temperature_change', 'humidity_change', 'temperature_abs_change', 'humidity_abs_change', 'temp_rolling_mean_5', 'temp_rolling_std_5', 'hum_rolling_mean_5', 'hum_rolling_std_5', 'temp_rolling_mean_15', 'temp_rolling_std_15', 'hum_rolling_mean_15', 'hum_rolling_std_15']  (Note: 14 features total, excluding timestamp, time_diff, session_id, label)Test labels distribution:0    361     6Name: label, dtype: int64

## 11. Summary### Feature Engineering Complete ✅### Raw Features (2):- `temperature` - sensor reading in °C- `humidity` - sensor reading in % RH### Engineered Features (12):| Feature | Description ||---------|-------------|temperature_change` | Current temp - previous temp (+ = increasedecrease) |humidity_change` | Current humidity - previous humidity |temperature_abs_change` | Absolute magnitude of temperature change |humidity_abs_change` | Absolute magnitude of humidity change |temp_rolling_mean_5` | Mean temperature of last 5 readings (within session) |temp_rolling_std_5` | Std deviation of last 5 temperature readings |hum_rolling_mean_5` | Mean humidity of last 5 readings |hum_rolling_std_5` | Std deviation of last 5 humidity readings |temp_rolling_mean_15` | Mean temperature of last 15 readings |temp_rolling_std_15` | Std deviation of last 15 temperature readings |hum_rolling_mean_15` | Mean humidity of last 15 readings |hum_rolling_std_15` | Std deviation of last 15 humidity readings |### Gap Handling:- Identified gaps > 10 seconds using `timestamp.diff()`- Created `session_id` that increments after each large gap- Rolling features computed **within each session** (never across gaps)### Missing Values:- Change features: 1 missing (first row)- Rolling window 5: 4 missing (first 4 rows of each session)- Rolling window 15: 14 missing (first 14 rows of each session)- Session boundaries add more missing values- **Dropped 32/17/14 rows** with missing engineered features (negligible)### Final Dataset Sizes:- Normal features: 36,920 rows × 14 features- Pilot features: 3,070 rows × 14 features- Test features: 42 rows × 14 features (6 anomaly, 36 normal)### Files Saved:- `data/processed/features/normal_features.csv`- `data/processed/features/pilot_features.csv`- `data/processed/features/test_features.csv`- `data/processed/features/test_labels.csv`---**Feature Engineering Complete.** Ready for Phase 4 (Model Training).